# Evaluate on the platform

Computing metrics by hand in a cell is the right move while you are inventing
one. `orinth.evaluate` is for the other case: the number you want the platform
to quote, recorded on `/testing`, comparable against every other evaluation of
the same split, with the per-item rows kept on disk.

In [ ]:
import orinth
import polars as pl

for entry in orinth.evaluate.datasets():
    mark = " " if entry["available"] else "x"
    print(f"[{mark}] {entry['key']:52} {entry['task_type']:20} {entry['split']}")

A `dataset_key` names a dataset **and a split** — `dataset:<id>:test`. That is
why `datasets()` exists rather than letting you build one: a key the platform
cannot resolve fails at job time, several seconds after you would like to know.

In [ ]:
target = next(entry for entry in orinth.evaluate.datasets() if entry["available"])
KEY = target["key"]

candidates = [
    model
    for model in orinth.models.list()
    if model.available and model.task_type == target["task_type"]
]
for model in candidates:
    print(f"{model.id:42} {model.family}")

MODEL_ID = candidates[0].id
print("\nscoring", MODEL_ID, "on", KEY)

## One model

`start()` waits by default — an evaluation is minutes, not hours, and the id is
not much use before the metrics land.

In [ ]:
def overall_of(metrics):
    # Per task: `image` for vision, `text_classification` for text. Found rather
    # than guessed, so this cell works on whichever the model is.
    for value in metrics.values():
        if isinstance(value, dict) and isinstance(value.get("overall"), dict):
            return value["overall"]
    return {}


job = orinth.evaluate.start(MODEL_ID, KEY, limit=50)
print(job.status, f"{job.percent:.0f}%", "|", job.error or "no error")

for key_name, value in overall_of(job.metrics).items():
    print(f"  {key_name:20} {value}")

## Several models, one comparison

`compare()` goes through the API's batch route rather than looping `start()`.
That is what groups the jobs under one `comparison_id` — a loop produces N
unrelated jobs the Testing page cannot line up side by side.

In [ ]:
if len(candidates) > 1:
    jobs = orinth.evaluate.compare([model.id for model in candidates[:3]], KEY, limit=50)
    print(
        pl.DataFrame(
            [
                {
                    "model": entry.model_id,
                    "status": entry.status,
                    **overall_of(entry.metrics),
                }
                for entry in jobs
            ]
        )
    )
else:
    print("Only one model for this task — train another and this cell ranks them.")

## The rows behind the number

This is where an aggregate becomes actionable. 0.8 accuracy is a fact about the
split; *these five items* is a fact you can do something about.

In [ ]:
rows = orinth.evaluate.per_item(job.id)
frame = pl.DataFrame(
    [
        {
            "item": row.get("image"),
            "truth": row.get("ground_truth"),
            "predicted": row.get("prediction"),
            "confidence": max((row.get("scores") or {}).values(), default=None),
        }
        for row in rows
    ],
    schema_overrides={"confidence": pl.Float64},
)
wrong = frame.filter(pl.col("truth") != pl.col("predicted"))
print(f"{wrong.height} of {frame.height} wrong")
wrong.sort("confidence", descending=True, nulls_last=True).head(10)

Confidently wrong, at the top. Those are either label errors in the dataset or
the exact failure mode worth fixing next — and the two are told apart by opening
the images, which is why the item names are here.